# LAB04 — Text Classification (SMS Spam)
Dataset: SMS Spam Collection (5572 tin, ham 86.6% / spam 13.4%). Pipeline: clean → stratified split (70/15/15) → fit TF-IDF/W2V **chỉ trên train** → LR/SVM → P/R/F1 + confusion matrix.

In [ ]:
from preprocessing import load_sms, stratified_split, fit_tfidf, document_average_vectors
df = load_sms('sms.tsv')
print(df.shape)
print(df['label'].value_counts())
print('dups:', df.duplicated().sum(), '| missing:', df['text'].isna().sum())
print(df['text'].str.len().describe())
tr, va, te = stratified_split(df)
print(len(tr), len(va), len(te), [round(v['label_id'].mean(),4) for v in (tr,va,te)])


Imbalance 86.6/13.4 → accuracy 86.6% ngay cả khi đoán mò ham; bắt buộc xem F1 minority class. Split stratified giữ đúng tỉ lệ ở cả 3 tập.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
vec, Xtr = fit_tfidf(tr['clean'].tolist(), ngram=(1,1))
print('vocab:', len(vec.vocabulary_))
lr = LogisticRegression(max_iter=1000).fit(Xtr, tr['label_id'])
for nm, dd in [('val',va),('test',te)]:
    p = lr.predict(vec.transform(dd['clean'].tolist()))
    print(nm, f"acc={accuracy_score(dd['label_id'],p):.4f} P={precision_score(dd['label_id'],p):.4f} R={recall_score(dd['label_id'],p):.4f} F1={f1_score(dd['label_id'],p):.4f}")


In [ ]:
import json, numpy as np
vocab = json.load(open('w2v_vocab.json'))
V = np.load('w2v_vectors.npy')  # Skip-gram NEG dim=50, train RIÊNG trên train-corpus (xem README)
emb = dict(zip(vocab, V))
Xtr_w = document_average_vectors(tr['clean'].tolist(), emb, 50)
Xte_w = document_average_vectors(te['clean'].tolist(), emb, 50)
lrw = LogisticRegression(max_iter=1000).fit(Xtr_w, tr['label_id'])
p = lrw.predict(Xte_w)
print(f"avgw2v+LR test acc={accuracy_score(te['label_id'],p):.4f} F1={f1_score(te['label_id'],p):.4f}")
svm = LinearSVC().fit(Xtr, tr['label_id'])
p2 = svm.predict(vec.transform(te['clean'].tolist()))
print(f"tfidf+SVM test acc={accuracy_score(te['label_id'],p2):.4f} F1={f1_score(te['label_id'],p2):.4f}")


Test F1: TF-IDF+LR 0.8962 vs AvgW2V+LR 0.8649 vs TF-IDF+SVM 0.9593. Spam là bài lexical → TF-IDF thắng; averaging làm loãng từ khóa hiếm (`num` +8.93 chìm trong mean). SVM tuyến tính tách sparse tốt hơn LR.

In [ ]:
# Exp5 n-gram + stopwords + leakage demo
v2, X2 = fit_tfidf(tr['clean'].tolist(), ngram=(1,2))
m2 = LogisticRegression(max_iter=1000).fit(X2, tr['label_id'])
p = m2.predict(v2.transform(te['clean'].tolist()))
print('uni+bi vocab:', len(v2.vocabulary_), f"F1={f1_score(te['label_id'],p):.4f} (uni: 0.8962)")
v3, X3 = fit_tfidf(tr['clean'].tolist(), stopwords='english')
m3 = LogisticRegression(max_iter=1000).fit(X3, tr['label_id'])
p = m3.predict(v3.transform(te['clean'].tolist()))
print('no-stopwords F1=', round(f1_score(te['label_id'],p),4), '(keep stopwords: 0.8962)')
from sklearn.feature_extraction.text import TfidfVectorizer
leak = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b').fit(df['clean'].tolist())
print(f"vocab leak(full-fit)={len(leak.vocabulary_)} vs clean(train-fit)={len(vec.vocabulary_)}")


In [ ]:
# Exp4 word order: averaging không thấy phủ định
def avg(s): return np.mean([emb[w] for w in s.split() if w in emb], axis=0)
a, b = avg('this movie is good'), avg('this movie is not good')
print('cos(S1,S2)=', round(float(a@b/(np.linalg.norm(a)*np.linalg.norm(b))),4), '→ gần như cùng vector: giới hạn BoW')
# Top features (interpretability)
names = np.array(vec.get_feature_names_out()); c = lr.coef_[0]
print('TOP SPAM:', list(zip(names[np.argsort(c)[-10:][::-1]], np.round(c[np.argsort(c)[-10:][::-1]],2))))
print('TOP HAM:', list(zip(names[np.argsort(c)[:10]], np.round(c[np.argsort(c)[:10]],2))))


In [ ]:
# Confusion matrix → confusion_matrix.png
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
p = lr.predict(vec.transform(te['clean'].tolist()))
cm = confusion_matrix(te['label_id'], p)
print(cm)  # [[TN,FP],[FN,TP]] = [[719,5],[17,95]]
fig, ax = plt.subplots()
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['pred ham','pred spam']); ax.set_yticklabels(['true ham','true spam'])
ax.set_title('TF-IDF + LR — test confusion matrix')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i,j], ha='center', va='center')
fig.tight_layout(); fig.savefig('confusion_matrix.png', dpi=150)


Kết luận: FN (17) >> FP (5) do imbalance → recall 0.848 < precision 0.95. Lỗi điển hình: spam 'sạch' không từ khóa, ham cực ngắn bị `num` đánh lừa. Chi tiết xem error_analysis.md.